In [ ]:
import asyncio
import json
import os
from typing import Annotated, Any, Never

from agent_framework import (
    AgentExecutor,
    AgentExecutorRequest,
    AgentExecutorResponse,
    Message,
    WorkflowBuilder,
    WorkflowContext,
    executor,
    tool,
)
from agent_framework.foundry import FoundryChatClient
from azure.identity import AzureCliCredential
from dotenv import load_dotenv
from IPython.display import HTML, display
from pydantic import BaseModel

print("✅ All imports successful!")


## ခြေလှမ်း ၁: ဖွဲ့စည်းထားသော အထွက်များအတွက် Pydantic မော်ဒယ်များ သတ်မှတ်ခြင်း

ဤမော်ဒယ်များသည် အေးဂျင့်များ ပြန်လည်ပေးမည့် **schema** ကို သတ်မှတ်သည်။ Pydantic ဖြင့် `response_format` ကို အသုံးပြုခြင်းက အောက်ပါအတိုင်း သေချာစေပါသည်-
- ✅ အမျိုးအစား-လုံခြုံသော ဒေတာ ယူခြင်း
- ✅ အလိုအလျောက် စစ်ဆေးမှု
- ✅ အခမဲ့စာသား အဖြေများမှ သို့ မဖြစ်သော အမှား မရှိခြင်း
- ✅ ကွက်ထားထားသောအချက်အလက်များအပေါ် မူတည်၍ လွယ်ကူသော အခြေအနေ အရ လမ်းညွှန်မှု


In [ ]:
class BookingCheckResult(BaseModel):
    """Result from checking hotel availability at a destination."""

    destination: str
    has_availability: bool
    message: str


class AlternativeResult(BaseModel):
    """Suggested alternative destination when no rooms available."""

    alternative_destination: str
    reason: str


class BookingConfirmation(BaseModel):
    """Booking suggestion when rooms are available."""

    destination: str
    action: str
    message: str


print("✅ Pydantic models defined:")
print("   - BookingCheckResult (availability check)")
print("   - AlternativeResult (alternative suggestion)")
print("   - BookingConfirmation (booking confirmation)")

## အဆင့် ၂: ဟိုတယ်ခွဲစာရင်းကိရိယာ ဖန်တီးခြင်း

ဒီကိရိယာက **availability_agent** ပြောကြားမှာဖြစ်ပြီး အခန်းများ ရနိုင်/မရနိုင် စစ်ဆေးဖို့အတွက် အသုံးပြုမှာပါ။ `@ai_function` decorator ကို ဒီအရာများအတွက် အသုံးပြုပါသည် -
- Python function ကို AI အသုံးပြုနိုင်သော ကိရိယာတစ်ခုအဖြစ် ပြောင်းလဲခြင်း
- LLM အတွက် JSON schema ကို မူလိပ်အလိုအလျောက် ဖန်တီးခြင်း
- ပါရာမီတာ စစ်ဆေးချက်များ ကိုင်တွယ်ခြင်း
- agent များမှ အလိုအလျောက် ခေါ်ယူနိုင်စေရန် ဖွင့်လှစ်ခြင်း

ဒီချုပ်ပြသမှုအတွက် -
- **Stockholm, Seattle, Tokyo, London, Amsterdam** → အခန်းများ ရရှိနိုင် ✅
- **အခြားမြို့များအားလုံး** → အခန်းများ မရှိပါ ❌


In [ ]:
@tool(description="Check hotel room availability for a destination city")
def hotel_booking(destination: Annotated[str, "The destination city to check for hotel rooms"]) -> str:
    """
    Simulates checking hotel room availability.

    Returns JSON string with availability status.
    """
    display(
        HTML(f"""
        <div style='padding: 15px; background: #e3f2fd; border-left: 4px solid #2196f3; border-radius: 4px; margin: 10px 0;'>
            <strong>🔍 Tool Invoked:</strong> hotel_booking("{destination}")
        </div>
    """)
    )

    # Simulate availability check
    cities_with_rooms = ["stockholm", "seattle", "tokyo", "london", "amsterdam"]
    has_rooms = destination.lower() in cities_with_rooms

    result = {"has_availability": has_rooms, "destination": destination}

    return json.dumps(result)


print("✅ hotel_booking tool created with @tool decorator")

## အဆင့် ၃: လမ်းကြောင်းသတ်မှတ်မှုအတွက် အခြေအနေဖောင်ရွက်ရှင်းများ သတ်မှတ်ခြင်း

ဤဖောင်ရွက်ရှင်းများသည် agent ၏ဖြေကြားချက်ကို စစ်ဆေးပြီး workflow တွင် မည်သည့်လမ်းကြောင်းကို သွားစေရမည်ကို ဆုံးဖြတ်သည်။

**အဓိကပုံစံ**
1. စကားဝိုင်းသည် `AgentExecutorResponse` ဖြစ်သည်မလား စစ်ဆေးခြင်း
2. ဖွဲ့စည်းပုံရှိ ထွက်ရှိမှုကို (Pydantic မော်ဒယ်) ပေါင်းထုတ်ခြင်း
3. လမ်းကြောင်းထိန်းချုပ်မှုအတွက် `True` သို့မဟုတ် `False` ပြန်ပေးခြင်း

workflow သည် ဒီအခြေအနေများကို **အဆက်အတန်းများ** ပေါ်မှာ သုံးသပ်ပြီး နောက်တစ်ခုဖြင့် ဖွင့်မည့် executor ကိုဆုံးဖြတ်သည်။


In [ ]:
def has_availability_condition(message: Any) -> bool:
    """
    Condition for routing when hotels ARE available.
    
    Returns True if the destination has hotel rooms.
    """
    if not isinstance(message, AgentExecutorResponse):
        return True  # Default to True if unexpected type

    try:
        result = BookingCheckResult.model_validate_json(message.agent_run_response.text)

        display(
            HTML(f"""
            <div style='padding: 12px; background: #c8e6c9; border-left: 4px solid #4caf50; border-radius: 4px; margin: 10px 0;'>
                <strong>✅ Condition Check:</strong> has_availability = <strong>{result.has_availability}</strong> for {result.destination}
            </div>
        """)
        )

        return result.has_availability
    except Exception as e:
        display(
            HTML(f"""
            <div style='padding: 12px; background: #ffcdd2; border-left: 4px solid #f44336; border-radius: 4px; margin: 10px 0;'>
                <strong>⚠️  Error:</strong> {str(e)}
            </div>
        """)
        )
        return False


def no_availability_condition(message: Any) -> bool:
    """
    Condition for routing when hotels are NOT available.
    
    Returns True if the destination has no hotel rooms.
    """
    if not isinstance(message, AgentExecutorResponse):
        return False

    try:
        result = BookingCheckResult.model_validate_json(message.agent_run_response.text)

        display(
            HTML(f"""
            <div style='padding: 12px; background: #ffecb3; border-left: 4px solid #ff9800; border-radius: 4px; margin: 10px 0;'>
                <strong>❌ Condition Check:</strong> no_availability for {result.destination}
            </div>
        """)
        )

        return not result.has_availability
    except Exception as e:
        return False


print("✅ Condition functions defined:")
print("   - has_availability_condition (routes when rooms exist)")
print("   - no_availability_condition (routes when no rooms)")

## အဆင့် ၄: စိတ်ကြိုက်ပြသမှု Executor ဖန်တီးခြင်း

Executors သည် ပြောင်းလဲမှုများ သို့မဟုတ် ဘက်ထရီ သက်ရောက်မှုများကို ဆောင်ရွက်သည့် workflow component များဖြစ်သည်။ နောက်ဆုံးရလဒ်ကို ပြသသည့် စိတ်ကြိုက် executor တစ်ခု ဖန်တီးရန် `@executor` decorator ကို အသုံးပြုသည်။

**အဓိကအကြောင်းအရာများ-**
- `@executor(id="...")` - function ကို workflow executor အဖြစ် မှတ်ပုံတင်သည်။
- `WorkflowContext[Never, str]` - input/output အတွက် Type hints
- `ctx.yield_output(...)` - နောက်ဆုံး workflow ရလဒ်ကို yield လုပ်သည်။


In [ ]:
@executor(id="display_result")
async def display_result(response: AgentExecutorResponse, ctx: WorkflowContext[Never, str]) -> None:
    """
    Display the final result as workflow output.
    
    This executor receives the final agent response and yields it as the workflow output.
    """
    display(
        HTML("""
        <div style='padding: 15px; background: #f3e5f5; border-left: 4px solid #9c27b0; border-radius: 4px; margin: 10px 0;'>
            <strong>📤 Display Executor:</strong> Yielding workflow output
        </div>
    """)
    )

    await ctx.yield_output(response.agent_run_response.text)


print("✅ display_result executor created with @executor decorator")

## အဆင့် ၅: ပတ်ဝန်းကျင် ပြောင်းလဲမှု များကို โหลดပါ

LLM client ကို ပြင်ဆင်ပါ။ ဤနမူနာသည် အောက်ပါအတိုင်း လုပ်ဆောင်သည်။
- **Microsoft Foundry** / **Azure OpenAI** (Responses API — အကြံပြုသည်)
- **Azure OpenAI**
- **OpenAI**


In [ ]:
# Load environment variables
load_dotenv()

# Configure the Microsoft Foundry provider with keyless authentication
provider = FoundryChatClient(
    project_endpoint=os.environ["AZURE_AI_PROJECT_ENDPOINT"],
    model=os.environ["AZURE_AI_MODEL_DEPLOYMENT_NAME"],
    credential=AzureCliCredential(),
)


## အဆင့် ၆: ဖွဲ့စည်းထားသော အထွက်များနှင့် AI အေးဂျင့်များ ဖန်တီးခြင်း

ကျနော်တို့ **အထူးပြု အေးဂျင့် သုံးယောက်** ကို `AgentExecutor` ဖြင့် ထုပ်ပိုးပြီး ဖန်တီးမည်ဖြစ်သည်။

၁။ **availability_agent** - လုပ်ငန်းသုံးကိရိယာဖြင့် ဟိုတယ် ရရှိနိုင်မှု စစ်ဆေးသည်
၂။ **alternative_agent** - အခန်း မရှိရင် အခြားမြို့များကို အကြံပြုသည်
၃။ **booking_agent** - အခန်း ရှိသော အခါ မှာကြားရန် တားမြစ်သည်

**အဓိက လက္ခဏာများ:**
- `tools=[hotel_booking]` - အေးဂျင့်အား လုပ်ငန်းသုံးကိရိယာ ပေးသည်
- `response_format=PydanticModel` - ဖွဲ့စည်းထားသော JSON အထွက်ထုတ်ရန် တောင်းဆိုသည်
- `AgentExecutor(..., id="...")` - အေးဂျင့်ကို လုပ်ငန်းစဉ် အသုံးပြုရန် ထုပ်ပိုးသည်


In [ ]:
# Agent 1: Check availability with tool
availability_agent = AgentExecutor(
    provider.as_agent(
        name="availability-agent",
        instructions=(
            "You are a hotel booking assistant that checks room availability. "
            "Use the hotel_booking tool to check if rooms are available at the destination. "
            "Return JSON with fields: destination (string), has_availability (bool), and message (string). "
            "The message should summarize the availability status."
        ),
        tools=[hotel_booking],
        default_options={"response_format": BookingCheckResult},
    ),
    id="availability_agent",
)

# Agent 2: Suggest alternative (when no rooms)
alternative_agent = AgentExecutor(
    provider.as_agent(
        name="alternative-agent",
        instructions=(
            "You are a helpful travel assistant. When a user cannot find hotels in their requested city, "
            "suggest an alternative nearby city that has availability. "
            "Return JSON with fields: alternative_destination (string) and reason (string). "
            "Make your suggestion sound appealing and helpful."
        ),
        default_options={"response_format": AlternativeResult},
    ),
    id="alternative_agent",
)

# Agent 3: Suggest booking (when rooms available)
booking_agent = AgentExecutor(
    provider.as_agent(
        name="booking-agent",
        instructions=(
            "You are a booking assistant. The user has found available hotel rooms. "
            "Encourage them to book by highlighting the destination's appeal. "
            "Return JSON with fields: destination (string), action (string), and message (string). "
            "The action should be 'book_now' and message should be encouraging."
        ),
        default_options={"response_format": BookingConfirmation},
    ),
    id="booking_agent",
)

display(
    HTML("""
    <div style='padding: 15px; background: #e3f2fd; border-left: 4px solid #2196f3; border-radius: 4px; margin: 10px 0;'>
        <strong>✅ Created 3 Agents:</strong>
        <ul style='margin: 10px 0 0 0;'>
            <li><strong>availability_agent</strong> - Checks availability with hotel_booking tool</li>
            <li><strong>alternative_agent</strong> - Suggests alternative cities</li>
            <li><strong>booking_agent</strong> - Encourages booking</li>
        </ul>
    </div>
""")
)


## လှုပ်ရှားမှု အဆင့် ၇: အခြေအနေ အရ လမ်းကြောင်းများဖြင့် Workflow ကို တည်ဆောက်ခြင်း

ယခုမှာ `WorkflowBuilder` ကို သုံးပြီး အခြေအနေ အရ လမ်းကြောင်းရှိ အစုလိုက်ကွန်ရက်ကို တည်ဆောက်မှာ ဖြစ်ပါတယ်။

**Workflow ဖွဲ့စည်းပုံ။**
```
availability_agent (START)
        ↓
   Evaluate conditions
        ↙         ↘
[no_availability]  [has_availability]
        ↓              ↓
alternative_agent  booking_agent
        ↓              ↓
    display_result ←───┘
```

**အဓိက နည်းလမ်းများ။**
- `.set_start_executor(...)` - ဖြင့် မောင်းနှင်မှု စတင်ရာနေရာ သတ်မှတ်သည်။
- `.add_edge(from, to, condition=...)` - အခြေအနေပါသော လမ်းကြောင်း ထည့်သည်။
- `.build()` - Workflow ကို အဆုံးသတ်သည်။


In [ ]:
# Build the workflow with conditional routing
workflow = (
    WorkflowBuilder(
        start_executor=availability_agent,
        output_executors=[display_result],
    )
    # NO AVAILABILITY PATH
    .add_edge(availability_agent, alternative_agent, condition=no_availability_condition)
    .add_edge(alternative_agent, display_result)
    # HAS AVAILABILITY PATH
    .add_edge(availability_agent, booking_agent, condition=has_availability_condition)
    .add_edge(booking_agent, display_result)
    .build()
)

display(
    HTML("""
    <div style='padding: 20px; background: linear-gradient(135deg, #667eea 0%, #764ba2 100%); color: white; border-radius: 8px; margin: 10px 0;'>
        <h3 style='margin: 0 0 15px 0;'>✅ Workflow Built Successfully!</h3>
        <p style='margin: 0; line-height: 1.6;'>
            <strong>Conditional Routing:</strong><br>
            • If <strong>NO availability</strong> → alternative_agent → display_result<br>
            • If <strong>availability</strong> → booking_agent → display_result
        </p>
    </div>
""")
)

## အဆင့် 8: စမ်းသပ်မှုခန်း 1 ကို ပြေးပါ - စတီး မြို့တွင် ရရှိနိုင်မှု မရှိခြင်း (ပါရီ)

ပါရီမြို့ရှိ ဟိုတယ်များ (ကျွန်ုပ်၏ ဆင်ဆာမှုတွင် အခန်းများမရှိပါ) ကို တောင်းဆိုခြင်းဖြင့် **ရရှိနိုင်မှု မရှိခြင်း** လမ်းကြောင်းကို စမ်းသပ်ကြည့်ပါမည်။


In [ ]:
display(
    HTML("""
    <div style='padding: 20px; background: #fff3e0; border-left: 4px solid #ff9800; border-radius: 8px; margin: 20px 0;'>
        <h3 style='margin: 0 0 10px 0; color: #e65100;'>🧪 TEST CASE 1: Paris (No Availability)</h3>
        <p style='margin: 0;'>Expected workflow path: availability_agent → alternative_agent → display_result</p>
    </div>
""")
)

# Create request for Paris
request_paris = AgentExecutorRequest(
    messages=[Message(role="user", contents=["I want to book a hotel in Paris"])], should_respond=True
)

# Run the workflow
events_paris = await workflow.run(request_paris)
outputs_paris = events_paris.get_outputs()

# Display results
if outputs_paris:
    result_paris = AlternativeResult.model_validate_json(outputs_paris[0])

    display(
        HTML(f"""
        <div style='padding: 25px; background: linear-gradient(135deg, #FFD700 0%, #FFA500 100%); border-radius: 12px; box-shadow: 0 4px 12px rgba(255,165,0,0.3); margin: 20px 0;'>
            <h3 style='margin: 0 0 15px 0; color: #333;'>🏆 WORKFLOW RESULT (Paris)</h3>
            <div style='background: white; padding: 20px; border-radius: 8px;'>
                <p style='margin: 0 0 10px 0; font-size: 16px;'><strong>Status:</strong> ❌ No rooms in Paris</p>
                <p style='margin: 0 0 10px 0; font-size: 16px;'><strong>Alternative Suggestion:</strong> 🏨 {result_paris.alternative_destination}</p>
                <p style='margin: 0; font-size: 14px; color: #666;'><strong>Reason:</strong> {result_paris.reason}</p>
            </div>
        </div>
    """)
    )

## အဆင့် ၉: စစ်ဆေးမှုအမှု ၂ ကို လုပ်ဆောင်ပါ - မြို့နှင့် အရနိုင်မှုပါဝင်သော (စတွောက်ဟော့မ်)

ယခု **အရနိုင်မှု** လမ်းကြောင်းကို စမ်းသပ်ကြည့်မည်၊ စတွောက်ဟော့မ်ရှိ ဟိုတယ်များကို တောင်းဆိုခြင်းဖြင့် (အဆိုပါနေရာတွင် အခန်းများ ရှိသည်။


In [ ]:
display(
    HTML("""
    <div style='padding: 20px; background: #e8f5e9; border-left: 4px solid #4caf50; border-radius: 8px; margin: 20px 0;'>
        <h3 style='margin: 0 0 10px 0; color: #1b5e20;'>🧪 TEST CASE 2: Stockholm (Has Availability)</h3>
        <p style='margin: 0;'>Expected workflow path: availability_agent → booking_agent → display_result</p>
    </div>
""")
)

# Create request for Stockholm
request_stockholm = AgentExecutorRequest(
    messages=[Message(role="user", contents=["I want to book a hotel in Stockholm"])], should_respond=True
)

# Run the workflow
events_stockholm = await workflow.run(request_stockholm)
outputs_stockholm = events_stockholm.get_outputs()

# Display results
if outputs_stockholm:
    result_stockholm = BookingConfirmation.model_validate_json(outputs_stockholm[0])

    display(
        HTML(f"""
        <div style='padding: 25px; background: linear-gradient(135deg, #4caf50 0%, #8bc34a 100%); color: white; border-radius: 12px; box-shadow: 0 4px 12px rgba(76,175,80,0.3); margin: 20px 0;'>
            <h3 style='margin: 0 0 15px 0;'>🏆 WORKFLOW RESULT (Stockholm)</h3>
            <div style='background: white; color: #333; padding: 20px; border-radius: 8px;'>
                <p style='margin: 0 0 10px 0; font-size: 16px;'><strong>Status:</strong> ✅ Rooms Available!</p>
                <p style='margin: 0 0 10px 0; font-size: 16px;'><strong>Destination:</strong> 🏨 {result_stockholm.destination}</p>
                <p style='margin: 0 0 10px 0; font-size: 16px;'><strong>Action:</strong> {result_stockholm.action}</p>
                <p style='margin: 0; font-size: 14px; color: #666;'><strong>Message:</strong> {result_stockholm.message}</p>
            </div>
        </div>
    """)
    )

## အရေးကြီးသောအချက်များနှင့် နောက်လုပ်ဆောင်ရန်အဆင့်များ

### ✅ သင်သိရှိရသောအရာများ:

1. **WorkflowBuilder ပုံစံ**
   - `.set_start_executor()` ကို အသုံးပြု၍ စတင်မှုနေရာ သတ်မှတ်ရန်
   - `.add_edge(from, to, condition=...)` ကို သတ်မှတ်ထားသော အခြေအနေဖြင့် လမ်းကြောင်းပေါ် သွားရန် အသုံးပြုရန်
   - `.build()` ကို ဖုန်းခေါ်၍ workflow ကို အဆုံးသတ်ရန်

2. **အခြေအနေ အခြေခံ လမ်းကြောင်းရွေးချယ်ခြင်း**
   - Condition function များသည် `AgentExecutorResponse` ကို ကြည့်ရှုသည်
   - ဖွဲ့စည်းထားသော output များကို ခွဲခြမ်းစိတ်ဖြာ၍ လမ်းကြောင်း တည်ဖြတ်ချက်ယူသည်
   - Edge ကို ဖွင့်ရန် `True` ကို ပြန်သွားပြီး၊ ကျော်ရန် `False` ကို ပြန်ပေးသည်

3. **ကိရိယာ ပေါင်းစည်းခြင်း**
   - Python function များကို AI ကိရိယာများ ဖြစ်စေရန် `@ai_function` ကို အသုံးပြုသည်
   - Agents များသည် လိုအပ်သည့်အခါ ကိရိယာများကို ကိုယ်တိုင် ခေါ်ယူသည်
   - ကိရိယာများက agents များ parse လုပ်နိုင်သော JSON ကို ပြန်ပေးသည်

4. **ဖွဲ့စည်းထားသော output များ**
   - Pydantic မော်ဒယ်များကို type-safe အချက်အလက်ယူရန် အသုံးပြုသည်
   - Agents ဖန်တီးရာတွင် `response_format=MyModel` ဟု သတ်မှတ်ရန်
   - `Model.model_validate_json()` ဖြင့် ပြန်လည် validate လုပ်ရန်

5. **စိတ်ကြိုက် Executors များ**
   - Workflow ကွန်ပေါင်းများဖန်တီးရန် `@executor(id="...")` ကို အသုံးပြုသည်
   - Executors များသည် အချက်အလက် ပြောင်းလဲခြင်း သို့မဟုတ် ဘေးထွက်ထွက်လုပ္ဆောင်ချက်များ ပြုလုပ်နိုင်သည်
   - Workflow ရလဒ်များ ထုတ်ယူရန် `ctx.yield_output()` ကို အသုံးပြုသည်

### 🚀 လက်တွေ့အသုံးချမှုများ:

- **ခရီးသွားဘွတ်ကင်**: ရရှိနိုင်မှုစစ်ဆေး၊ ဖြစ်နိုင်ခြေ အကြံပြု၊ ရွေးချယ်မှုများ နှိုင်းယှဉ်
- **ဖောက်သည်ဝန်ဆောင်မှု**: ပြဿနာအမျိုးအစား၊ ခံစားမှု၊ ဦးစားပေးမှု အပေါ် အခြေခံ၍ လမ်းကြောင်း
- **အီလက်ထရွနစ်ကုန်သည်**: အရစ်အတွက် စစ်ဆေး၊ ဖြစ်နိုင်ခြေ အကြံပြု၊ အမှာစာများ ကိုင်တွယ်
- **အကြောင်းအရာထိန်းချုပ်ခြင်း**: အသုံးပြုသူ အဆင်သင့် အချက်ပြချက်များနှင့် အဆင့်အတန်းအား ထောက်ခံ၍ လမ်းကြောင်း
- **ခွင့်ပြု ဆောင်ရွက်ဖွဲ့စည်းမှုများ**: ငွေပမာဏ၊ အသုံးပြုသူ အခန်းကဏ္ဍ၊ အန္တရာယ်အဆင့် အပေါ် အခြေခံ၍ လမ်းစဉ်
- **အဆင့်စုံ ပြုလုပ်ခြင်း**: အချက်အလက်အရည်အသွေး၊ ပြည့်စုံမှုအပေါ် မူတည်၍ လမ်းကြောင်းရွေးချယ်ခြင်း

### 📚 နောက်လုပ်ဆောင်ရန်အဆင့်များ:

- ပိုမိုရှုပ်ထွေးသော အခြေအနေများ (criteria များစွာ) ထည့်သွင်း
- workflow အခြေအနေ စီမံခန့်ခွဲမှုဖြင့် loop များ ထည့်သွင်း
- ထပ်ခါတလဲလဲ အသုံးပြုနိုင်သည့် sub-workflow များ ထည့်သွင်း
- လက်တွေ့ API များနှင့် ပေါင်းစည်းခြင်း (ဟိုတယ်ဘွတ်ကင်၊ ကုန်တိုက်စနစ်များ)
- အမှားကိုင်တွယ်မှုနှင့် fallback လမ်းကြောင်းများ ထည့်သွင်း
- အတွင်းပါ visualisation ကိရိယာများဖြင့် workflow များကို မြင်သာတင်ပြ


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**ပြောကြားချက်**
ဤစာတမ်းကို AI ဘာသာပြန်ဝန်ဆောင်မှု [Co-op Translator](https://github.com/Azure/co-op-translator) အသုံးပြု၍ ဘာသာပြန်ထားပါသည်။ ကျွန်ုပ်တို့သည် တိကျမှန်ကန်မှုအတွက် ကြိုးပမ်းနေသော်လည်း၊ စက်ကိရိယာဘာသာပြန်ခြင်းများတွင် အမှားများ သို့မဟုတ် မှားယွင်းချက်များ ပါဝင်နိုင်ကြောင်း သတိပြုပါရန် လိုအပ်ပါသည်။ မူလစာတမ်းကို မူရင်းဘာသာဖြင့်သာ ယုံကြည်စိတ်ချရသော အချက်အလက်အဖြစ် သတ်မှတ်သင့်သည်။ အရေးကြီးသည့် သတင်းအချက်အလက်များအတွက် ပရော်ဖက်ရှင်နယ် လူသားဘာသာပြန်သူဝန်ဆောင်မှုကို အကြံပြုပါသည်။ ဤဘာသာပြန်ချက်ကို အသုံးပြုခြင်းမှ ဖြစ်ပေါ်လာသော နားလည်မှုကွာခြားမှုများ သို့မဟုတ် မမှန်ကန်သော အသုံးပြုမှုများအတွက် ကျွန်ုပ်တို့ တာဝန်မခံပါ။
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
